# 🌊 V20 Colossus Ensemble — Flood Risk Prediction

**Competition:** ML Opsidian Genesis — Flood Risk Prediction (Sri Lanka)  
**Version:** V20 Colossus (The 185-Model Multi-GPU Swarm)  
**Ultimate Final Score:** `0.38125` (Champion)  


---

## 🏆 What Makes Colossus Different

This notebook represents the culmination of every modeling insight gathered across 19 prior versions. The key architectural decisions are backed by hard empirical evidence from the leaderboard:

| Decision | Why |
|---|---|
| **Raw target prediction** | V19 proved Logit transform distorts optimization landscape |
| **No Pseudo-Labeling** | V19 proved it creates confirmation bias on test set |
| **100-Fold CV** (5×20) | Maximizes stability — reduces seed-level variance to near zero |
| **10 random seeds per config** | Seed averaging neutralizes random initialization variance |
| **Deep CatBoost (d8-d12)** | Proven essential for Explained Variance Penalty in custom metric |
| **HuberRegressor (ε=1.35) stacker** | Mathematically clamps outliers while preserving variance tracking |
| **Checkpointing** | Every model saved to disk — fully crash-safe over multi-day runs |

## 📊 Custom Evaluation Metric
The leaderboard uses a **two-component metric** (lower is better):
1. **Balanced Error Assessment** — Penalizes outlier predictions harshly
2. **Explained Variance Penalty** — Scales the error by how well predictions *track* fluctuations

> ⚠️ **The OOF Overfitting Paradox**: Models optimized purely for OOF RMSE (shallow/smooth models like HGB) achieve great local scores but fail the Explained Variance Penalty on the leaderboard. We must include deep trees (Cat depth 8-12) even if they increase OOF RMSE slightly.

## 📦 1. Imports & Configuration

We use four algorithm families to maximize diversity:
- **HistGradientBoosting** — Fast, smooth anchor models
- **LightGBM** — MAE, RMSE, Deep, Wide, Huber, DART variants
- **XGBoost** — SquaredError and PseudoHuber objective variants  
- **CatBoost** — Depths 3–12, slow LR variants, high regularization variants

Key constants:
- `N_FOLDS = 20`, `N_REPEATS = 5` → **100 folds per model**
- `N_SEEDS = 10` → Seeds: 42, 142, 242, ... 942

In [ ]:
import numpy as np
import pandas as pd
import os
import warnings
import time
import gc
warnings.filterwarnings('ignore')

from sklearn.model_selection import RepeatedKFold, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

import lightgbm as lgb
from catboost import CatBoostRegressor

try:
    import xgboost as xgb
    HAS_XGB = True
    print("✅ XGBoost available")
except ImportError:
    HAS_XGB = False
    print("⚠️  XGBoost not installed — skipping. Run: pip install xgboost")

# ─── Configuration ────────────────────────────────────
SEED           = 42
N_FOLDS        = 20
N_REPEATS      = 5          # 5 × 20 = 100 folds per model
DATA_DIR       = "data"
CHECKPOINT_DIR = "colossus_checkpoints"
N_SEEDS        = 10
ALL_SEEDS      = [42, 142, 242, 342, 442, 542, 642, 742, 842, 942]
TOTAL_FOLDS    = N_FOLDS * N_REPEATS

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
np.random.seed(SEED)

print(f"""\n{'='*60}
  V20 COLOSSUS CONFIGURATION
  CV:         {N_FOLDS} folds × {N_REPEATS} repeats = {TOTAL_FOLDS} folds/model
  Seeds:      {N_SEEDS} per config
  Checkpoint: {CHECKPOINT_DIR}/
{'='*60}""")

## 📂 2. Data Loading

- **Train:** 20,886 rows × 47 features
- **Test:** 5,300 rows × 46 features  
- **Target:** `flood_risk_score` ∈ [0, 1] — continuous flood risk probability

In [ ]:
print("Loading data...")
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

TARGET = 'flood_risk_score'
test_record_ids = test['record_id'].copy()

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
print(f"\nTarget distribution:")
print(train[TARGET].describe().round(4))

## 🔧 3. Feature Engineering

### 3a. Temporal & Meta Features
Extract seasonality signals from `generation_date`. Sri Lanka has two monsoon seasons:
- **NE Monsoon** (Dec–Feb): Heavy rainfall in the north/east
- **SW Monsoon** (May–Sep): Heavy rainfall in the south/west

We encode month as **sine/cosine** to preserve cyclical continuity (month 12 is adjacent to month 1).

We also parse the `reason_not_good_to_live` free-text field into structured binary flags.

In [ ]:
for df in [train, test]:
    df['gen_date']        = pd.to_datetime(df['generation_date'])
    df['gen_month']       = df['gen_date'].dt.month
    df['gen_year']        = df['gen_date'].dt.year
    df['gen_day_of_year'] = df['gen_date'].dt.dayofyear
    df['gen_quarter']     = df['gen_date'].dt.quarter
    df['is_ne_monsoon']   = df['gen_month'].isin([12, 1, 2]).astype(int)
    df['is_sw_monsoon']   = df['gen_month'].isin([5, 6, 7, 8, 9]).astype(int)
    df['gen_month_sin']   = np.sin(2 * np.pi * df['gen_month'] / 12)
    df['gen_month_cos']   = np.cos(2 * np.pi * df['gen_month'] / 12)
    reason = df['reason_not_good_to_live'].fillna('Other')
    df['reason_flood_flag'] = reason.str.contains('flood', case=False).astype(int)
    df['reason_infra_flag'] = reason.str.contains('infrastructure', case=False).astype(int)
    df['reason_road_flag']  = reason.str.contains('road', case=False).astype(int)
    df['reason_other_flag'] = (reason == 'Other').astype(int)
    df['is_good_binary']    = (df['is_good_to_live'] == 'Yes').astype(int)
    df['log_inundation']    = np.log1p(df['inundation_area_sqm'])
    df['sqrt_inundation']   = np.sqrt(df['inundation_area_sqm'])
    df['inundation_per_pop']= df['inundation_area_sqm'] / (df['population_density_per_km2'] + 1)
    df['record_id_num']     = df['record_id'].str.replace('F', '', regex=False).astype(int)

print("✅ Temporal and meta features added")

### 3b. Domain-Driven Interaction Features

We create **physically meaningful interaction terms** based on hydrological domain knowledge:

| Feature | Formula | Why It Matters |
|---|---|---|
| `river_rain_risk` | `rainfall / (river_distance + 1)` | Proximity + intensity = flood hazard |
| `composite_vuln` | Weighted sum of rainfall + floods + drainage | Single vulnerability score |
| `urban_runoff` | `built_up_pct × rainfall / 100` | Urban surfaces prevent water absorption |
| `bad_drainage_rain` | `(drainage < 0.35) × rainfall` | Flag high-risk low-drainage + heavy rain |
| `elev_rain_ratio` | `rainfall / (elevation + 1)` | Low elevation with heavy rain = extreme risk |

In [ ]:
def engineer_features(df):
    df = df.copy(); eps = 1e-6
    df['rainfall_x_flood']       = df['rainfall_7d_mm'] * df['historical_flood_count']
    df['monthly_x_flood']        = df['monthly_rainfall_mm'] * df['historical_flood_count']
    df['rain_ratio']              = df['rainfall_7d_mm'] / (df['monthly_rainfall_mm'] + eps)
    df['rain_cum']                = df['rainfall_7d_mm'] + df['monthly_rainfall_mm']
    df['river_clip']              = df['distance_to_river_m'].clip(lower=0)
    df['river_rain_risk']         = df['rainfall_7d_mm'] / (df['river_clip'] + 1)
    df['river_monthly_risk']      = df['monthly_rainfall_mm'] / (df['river_clip'] + 1)
    df['elev_clip']               = df['elevation_m'].clip(lower=0)
    df['elev_rain_ratio']         = df['rainfall_7d_mm'] / (df['elev_clip'] + 1)
    df['low_elev_flag']           = (df['elevation_m'] < 30).astype(int)
    df['infra_socio']             = df['infrastructure_score'] * df['socioeconomic_status_index']
    df['water_veg_balance']       = df['ndwi'] - df['ndvi']
    df['ndvi_ndwi_product']       = df['ndvi'] * df['ndwi']
    df['ndwi_sq']                 = df['ndwi'] ** 2
    df['drainage_x_rain']         = df['drainage_index'] * df['rainfall_7d_mm']
    df['bad_drainage_rain']       = (df['drainage_index'] < 0.35).astype(int) * df['rainfall_7d_mm']
    df['urban_runoff']            = df['built_up_percent'] * df['rainfall_7d_mm'] / 100
    df['evac_hosp_sum']           = df['nearest_hospital_km'] + df['nearest_evac_km']
    df['max_dist_help']           = df[['nearest_hospital_km', 'nearest_evac_km']].max(axis=1)
    df['pop_x_rain']              = df['population_density_per_km2'] * df['rainfall_7d_mm']
    df['pop_x_flood']             = df['population_density_per_km2'] * df['historical_flood_count']
    df['extreme_x_rain']          = df['extreme_weather_index'] * df['rainfall_7d_mm']
    df['extreme_x_flood']         = df['extreme_weather_index'] * df['historical_flood_count']
    df['extreme_x_monthly']       = df['extreme_weather_index'] * df['monthly_rainfall_mm']
    df['seasonal_rain']           = df['seasonal_index'] * df['rainfall_7d_mm']
    df['seasonal_extreme']        = df['seasonal_index'] * df['extreme_weather_index']
    df['terrain_rain']            = df['terrain_roughness_index'] * df['rainfall_7d_mm']
    df['inundation_x_rain']       = df['inundation_area_sqm'] * df['rainfall_7d_mm']
    df['log_inundation_x_extreme']= df['log_inundation'] * df['extreme_weather_index']
    df['composite_vuln']          = (
        df['rainfall_7d_mm'] * 0.3 + df['historical_flood_count'] * 15.0 +
        df['extreme_weather_index'] * 50.0 + (1 - df['drainage_index']) * 30.0 +
        df['built_up_percent'] * 0.10)
    return df

train = engineer_features(train)
test  = engineer_features(test)
print(f"✅ Domain interaction features added")
print(f"   Train shape after engineering: {train.shape}")

### 3c. Categorical Encoding & Feature Matrix

We use **Label Encoding** for tree-based models (they handle ordinal integers naturally).  
High-cardinality columns like `place_name` and `district` will receive fold-safe Target Encoding in the next step.

We concatenate train+test before encoding to ensure consistent label mappings.

In [ ]:
CAT_COLS = ['district','landcover','soil_type','water_supply','electricity',
            'road_quality','urban_rural','water_presence_flag',
            'flood_occurrence_current_event','is_good_to_live',
            'reason_not_good_to_live','place_name']
DROP_COLS = ['record_id','gen_date','generation_date','is_synthetic', TARGET]

all_data = pd.concat([train, test], axis=0, ignore_index=True)
for col in CAT_COLS:
    if col in all_data.columns:
        le = LabelEncoder()
        all_data[col] = le.fit_transform(all_data[col].astype(str).fillna('missing'))

n_train   = len(train)
train_enc = all_data.iloc[:n_train].copy()
test_enc  = all_data.iloc[n_train:].copy()
train_enc[TARGET] = train[TARGET].values

EXCLUDE      = set(DROP_COLS + [TARGET])
feature_cols = [c for c in train_enc.columns if c not in EXCLUDE]
X      = train_enc[feature_cols].copy()
y      = train_enc[TARGET].copy()
X_test = test_enc[feature_cols].copy()

medians = X.median(numeric_only=True)
X       = X.fillna(medians)
X_test  = X_test.fillna(medians)

print(f"✅ Categorical encoding complete")
print(f"   Base features before Target Encoding: {X.shape[1]}")

### 3d. Fold-Safe Target Encoding

**Target Encoding** replaces a category with the mean target value for that category. However, naive target encoding causes severe **data leakage** — the model sees the answer during training.

**Fold-Safe TE** solves this by:
1. Split training data into K folds
2. For each fold's validation rows, compute the mean using **only the other K-1 folds' rows**
3. For test data, use the mean computed from ALL training rows (no leakage risk)
4. Apply **Bayesian smoothing**: `enc = (count × mean + smooth × global_mean) / (count + smooth)` — rare categories are shrunk towards the global mean

We apply this to: `place_name`, `district`, `soil_type`, `landcover`, `road_quality`, `flood_occurrence_current_event`

We then create **TE × raw feature interaction terms** (e.g., district mean risk × 7-day rainfall = district-specific rain risk).

**The result is our 115-feature set** — proven optimal across V11 through V20.

> 💡 **Caching**: This computation takes ~2-3 minutes. We cache the result to `colossus_checkpoints/features_cached.npz` so restarts are instant.

In [ ]:
TE_CACHE_FILE = os.path.join(CHECKPOINT_DIR, "features_cached.npz")

if os.path.exists(TE_CACHE_FILE):
    print("Loading cached features (skip recomputation)...")
    data   = np.load(TE_CACHE_FILE)
    X_arr      = data['X_arr']
    X_test_arr = data['X_test_arr']
    y_arr      = data['y_arr']
    print(f"✅ Loaded from cache | Feature count: {X_arr.shape[1]}")
else:
    def fold_safe_te(X_df, y_s, X_te_df, col, smooth, gm, n_folds=20, seed=SEED):
        """Compute fold-safe Bayesian target encoding."""
        tr_te = np.zeros(len(X_df))
        kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
        for tr_idx, val_idx in kf.split(X_df):
            s = (pd.DataFrame({'k': X_df.iloc[tr_idx][col].values,
                               'y': y_s.iloc[tr_idx].values})
                 .groupby('k')['y'].agg(['mean','count']))
            s['enc'] = (s['mean']*s['count'] + gm*smooth) / (s['count']+smooth)
            tr_te[val_idx] = X_df.iloc[val_idx][col].map(s['enc']).fillna(gm).values
        s_all = (pd.DataFrame({'k': X_df[col].values, 'y': y_s.values})
                 .groupby('k')['y'].agg(['mean','count']))
        s_all['enc'] = (s_all['mean']*s_all['count'] + gm*smooth) / (s_all['count']+smooth)
        return tr_te, X_te_df[col].map(s_all['enc']).fillna(gm).values

    gm = y.mean()
    print(f"Computing fold-safe target encodings (global mean = {gm:.4f})...")
    te_store = {}
    for col, smooth in [('place_name',5),('district',10),('soil_type',10),
                        ('landcover',10),('road_quality',10),
                        ('flood_occurrence_current_event',10)]:
        tr_te, te_te = fold_safe_te(X, y, X_test, col, smooth, gm)
        X[f'{col}_te'] = tr_te; X_test[f'{col}_te'] = te_te
        te_store[col]  = (tr_te, te_te)
        print(f"  ✅ {col}")

    # TE × raw feature interactions
    d_tr, d_te = te_store['district']
    f_tr, f_te = te_store['flood_occurrence_current_event']
    s_tr, s_te = te_store['soil_type']

    interactions = [
        ('rainfall_7d_mm',        d_tr, d_te, 'dist_te_x_rain'),
        ('extreme_weather_index', d_tr, d_te, 'dist_te_x_extreme'),
        ('historical_flood_count',d_tr, d_te, 'dist_te_x_flood'),
        ('rainfall_7d_mm',        f_tr, f_te, 'fl_te_x_rain'),
        ('extreme_weather_index', f_tr, f_te, 'fl_te_x_extreme'),
        ('drainage_index',        d_tr, d_te, 'dist_te_x_drainage'),
        ('river_clip',            d_tr, d_te, 'dist_te_x_river'),
        ('elevation_m',           d_tr, d_te, 'dist_te_x_elev'),
        ('historical_flood_count',f_tr, f_te, 'fl_te_x_flood'),
        ('drainage_index',        f_tr, f_te, 'fl_te_x_drainage'),
        ('rainfall_7d_mm',        s_tr, s_te, 'soil_te_x_rain'),
        ('extreme_weather_index', s_tr, s_te, 'soil_te_x_extreme'),
    ]
    for feat, tr_v, te_v, col_name in interactions:
        X[col_name]      = tr_v * X[feat]
        X_test[col_name] = te_v * X_test[feat]

    X_arr      = X.values.astype(np.float32)
    X_test_arr = X_test.values.astype(np.float32)
    y_arr      = y.values.astype(np.float32)
    np.savez(TE_CACHE_FILE, X_arr=X_arr, X_test_arr=X_test_arr, y_arr=y_arr)
    print(f"\n✅ Features cached to disk | Final feature count: {X_arr.shape[1]}")

print(f"\nFinal feature matrix: {X_arr.shape}")
print(f"Test matrix:          {X_test_arr.shape}")
print(f"Target range:         [{y_arr.min():.4f}, {y_arr.max():.4f}]")

## 🏗️ 4. Training Engine

### 4a. The 100-Fold Repeated CV Engine

The `train_and_cache` function is the heart of Colossus. It:

1. **Checks if checkpoints exist** — if `{name}_oof.npy` and `{name}_test.npy` already exist, it skips and prints `[SKIP]`. This makes every run fully resumable after a crash.
2. **Runs 100-fold Repeated KFold** — `RepeatedKFold(n_splits=20, n_repeats=5)` generates 100 train/val splits
3. **Averages OOF predictions** — Because repeated CV means each sample appears in validation 5 times (once per repeat), we sum predictions and divide by count
4. **Saves checkpoints** — Two `.npy` files per model for crash-safe recovery

In [ ]:
rkf = RepeatedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS, random_state=SEED)

def train_and_cache(name, train_fn):
    """100-Fold crash-safe training with automatic checkpointing."""
    oof_path  = os.path.join(CHECKPOINT_DIR, f"{name}_oof.npy")
    test_path = os.path.join(CHECKPOINT_DIR, f"{name}_test.npy")
    
    if os.path.exists(oof_path) and os.path.exists(test_path):
        print(f"[SKIP] '{name}' — checkpoint found, loading.")
        return
    
    print(f"\n{'='*55}")
    print(f" Model: {name}  |  {TOTAL_FOLDS} folds")
    print(f"{'='*55}")
    start_time = time.time()
    
    oof_preds_sum = np.zeros(len(X_arr))
    oof_counts    = np.zeros(len(X_arr))
    test_preds    = np.zeros(len(X_test_arr))
    
    for fold, (tr_i, val_i) in enumerate(rkf.split(X_arr)):
        X_tr, y_tr = X_arr[tr_i], y_arr[tr_i]
        X_va, y_va = X_arr[val_i], y_arr[val_i]
        
        preds_va, preds_te = train_fn(X_tr, y_tr, X_va, y_va, X_test_arr)
        oof_preds_sum[val_i] += preds_va
        oof_counts[val_i]    += 1
        test_preds           += preds_te / TOTAL_FOLDS
        
        # Print summary every 20 folds (once per repeat)
        if (fold + 1) % N_FOLDS == 0:
            current_oof = oof_preds_sum / np.maximum(oof_counts, 1)
            rmse_so_far = np.sqrt(mean_squared_error(y_arr, current_oof))
            elapsed = (time.time() - start_time) / 60
            print(f"  Repeat {(fold+1)//N_FOLDS}/{N_REPEATS} | Running RMSE={rmse_so_far:.5f} | {elapsed:.1f} min")
        
        gc.collect()
    
    final_oof  = oof_preds_sum / oof_counts
    rmse_total = np.sqrt(mean_squared_error(y_arr, final_oof))
    mins = (time.time() - start_time) / 60
    print(f"\n✅ DONE: {name} | OOF RMSE = {rmse_total:.5f} | {mins:.1f} min")
    
    np.save(oof_path,  final_oof)
    np.save(test_path, test_preds)

print("✅ Training engine ready")

### 4b. Model Trainer Factories

Each factory returns a `train_fn(X_tr, y_tr, X_va, y_va, X_te)` closure — a standard interface used by `train_and_cache`.

Key training decisions:
- **Early stopping** set to 150 rounds for LGB/XGB/CatBoost — prevents over-training
- **5000 max iterations** with early stopping — models stop well before this in practice  
- **tree_method='hist'** for XGBoost — uses the same histogram algorithm as LGB for speed

In [ ]:
def get_lgb_trainer(params):
    """LightGBM trainer with early stopping."""
    def train_fn(X_tr, y_tr, X_va, y_va, X_te):
        m = lgb.train(params,
                      lgb.Dataset(X_tr, y_tr),
                      num_boost_round=5000,
                      valid_sets=[lgb.Dataset(X_va, y_va)],
                      callbacks=[lgb.early_stopping(150, verbose=False)])
        return m.predict(X_va), m.predict(X_te)
    return train_fn

def get_cat_trainer(params):
    """CatBoost trainer. The workhorse of variance tracking."""
    def train_fn(X_tr, y_tr, X_va, y_va, X_te):
        m = CatBoostRegressor(**params)
        m.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True, verbose=0)
        return m.predict(X_va), m.predict(X_te)
    return train_fn

def get_xgb_trainer(params):
    """XGBoost trainer with DMatrix API for speed."""
    def train_fn(X_tr, y_tr, X_va, y_va, X_te):
        dtr = xgb.DMatrix(X_tr, label=y_tr)
        dva = xgb.DMatrix(X_va, label=y_va)
        dte = xgb.DMatrix(X_te)
        m = xgb.train(params, dtr,
                      num_boost_round=5000,
                      evals=[(dva, 'val')],
                      early_stopping_rounds=150,
                      verbose_eval=False)
        return m.predict(dva), m.predict(dte)
    return train_fn

def get_hgb_trainer(params):
    """HistGradientBoosting trainer (scikit-learn). Fast, smooth anchor models."""
    def train_fn(X_tr, y_tr, X_va, y_va, X_te):
        m = HistGradientBoostingRegressor(**params)
        m.fit(X_tr, y_tr)
        return m.predict(X_va), m.predict(X_te)
    return train_fn

print("✅ Trainer factories ready")

## 🚀 5. Phase 1 — HistGradientBoosting

**40 models** (4 depth configs × 10 seeds). Fastest phase, completes in ~2-4 hours.  
These produce conservative, smooth predictions that serve as stable **anchor models** for the Huber stacker.  

> ⚠️ **Important**: Due to the OOF Overfitting Paradox, HGB models alone produce misleadingly low OOF RMSE. They MUST be combined with deep CatBoost models in the final stack to satisfy the Explained Variance Penalty.

In [ ]:
print("[PHASE 1] HistGradientBoosting — 40 models")

hgb_configs = [
    ("hgb_d4",  dict(max_iter=800, learning_rate=0.03, max_depth=4,  early_stopping=True, validation_fraction=0.1)),
    ("hgb_d6",  dict(max_iter=800, learning_rate=0.03, max_depth=6,  early_stopping=True, validation_fraction=0.1)),
    ("hgb_d8",  dict(max_iter=800, learning_rate=0.03, max_depth=8,  early_stopping=True, validation_fraction=0.1)),
    ("hgb_d10", dict(max_iter=800, learning_rate=0.02, max_depth=10, early_stopping=True, validation_fraction=0.1)),
]

for cfg_name, cfg_params in hgb_configs:
    for s in ALL_SEEDS:
        p = cfg_params.copy(); p['random_state'] = s
        train_and_cache(f"{cfg_name}_s{s}", get_hgb_trainer(p))

## 🚀 6. Phase 2 — LightGBM

**60 models** (6 configs × 10 seeds). Moderate speed, ~8-12 hours.  

LightGBM configurations explore:
- **MAE objective** — Optimizes median, robust to outliers at the sample level
- **RMSE objective** — Standard mean-squared error optimization  
- **Deep (255 leaves)** — High-capacity model for variance tracking
- **Wide (127 leaves)** — Balanced mid-range option
- **Huber objective** — Native Huber loss in LGB for robust regression
- **DART** — Dropout Additive Regression Trees. Slower but highly accurate ensemble member

In [ ]:
print("[PHASE 2] LightGBM — 60 models")

lgb_base = {'metric':'rmse', 'n_jobs':-1, 'verbose':-1}

lgb_configs = [
    ("lgb_mae",  {**lgb_base, 'objective':'regression_l1', 'learning_rate':0.03,
                  'num_leaves':63,  'min_child_samples':20, 'feature_fraction':0.7,
                  'bagging_fraction':0.8, 'bagging_freq':5, 'reg_alpha':0.1, 'reg_lambda':1.0}),
    ("lgb_rmse", {**lgb_base, 'objective':'regression',    'learning_rate':0.03,
                  'num_leaves':63,  'min_child_samples':20, 'feature_fraction':0.7,
                  'bagging_fraction':0.8, 'bagging_freq':5, 'reg_alpha':0.1, 'reg_lambda':1.0}),
    ("lgb_deep", {**lgb_base, 'objective':'regression',    'learning_rate':0.02,
                  'num_leaves':255, 'min_child_samples':30, 'feature_fraction':0.6,
                  'bagging_fraction':0.75,'bagging_freq':5, 'reg_alpha':0.2, 'reg_lambda':2.0}),
    ("lgb_wide", {**lgb_base, 'objective':'regression',    'learning_rate':0.025,
                  'num_leaves':127, 'min_child_samples':25, 'feature_fraction':0.65,
                  'bagging_fraction':0.8, 'bagging_freq':5, 'reg_alpha':0.15,'reg_lambda':1.5}),
    ("lgb_huber",{**lgb_base, 'objective':'huber',         'learning_rate':0.03,
                  'num_leaves':63,  'min_child_samples':20, 'feature_fraction':0.7,
                  'bagging_fraction':0.8, 'bagging_freq':5, 'reg_alpha':0.1, 'reg_lambda':1.0}),
    ("lgb_dart", {**lgb_base, 'objective':'regression', 'boosting_type':'dart', 'learning_rate':0.05,
                  'num_leaves':63,  'min_child_samples':20, 'feature_fraction':0.7,
                  'bagging_fraction':0.8, 'bagging_freq':5, 'reg_alpha':0.1, 'reg_lambda':1.0}),
]

for cfg_name, cfg_params in lgb_configs:
    for s in ALL_SEEDS:
        p = cfg_params.copy(); p['seed'] = s
        train_and_cache(f"{cfg_name}_s{s}", get_lgb_trainer(p))

## 🚀 7. Phase 3 — XGBoost

**40 models** (4 configs × 10 seeds). Moderate speed, ~6-10 hours.  

XGBoost introduces genuine **algorithmic diversity** — its split-finding logic and regularization scheme differ fundamentally from LightGBM despite both being gradient boosting implementations.

- **SquaredError (d6/d8)** — Standard RMSE optimization
- **PseudoHuber (d6/d8)** — XGBoost's native robust loss, combining properties of MSE and MAE

In [ ]:
if HAS_XGB:
    print("[PHASE 3] XGBoost — 40 models")
    
    xgb_configs = [
        ("xgb_sq_d6",  {'objective':'reg:squarederror',    'eval_metric':'rmse', 'learning_rate':0.03,
                         'max_depth':6, 'subsample':0.8, 'colsample_bytree':0.7,
                         'alpha':0.1, 'lambda':1.0, 'tree_method':'hist'}),
        ("xgb_sq_d8",  {'objective':'reg:squarederror',    'eval_metric':'rmse', 'learning_rate':0.03,
                         'max_depth':8, 'subsample':0.8, 'colsample_bytree':0.7,
                         'alpha':0.1, 'lambda':1.0, 'tree_method':'hist'}),
        ("xgb_hub_d6", {'objective':'reg:pseudohubererror','eval_metric':'rmse', 'learning_rate':0.03,
                         'max_depth':6, 'subsample':0.8, 'colsample_bytree':0.7,
                         'alpha':0.1, 'lambda':1.0, 'tree_method':'hist'}),
        ("xgb_hub_d8", {'objective':'reg:pseudohubererror','eval_metric':'rmse', 'learning_rate':0.03,
                         'max_depth':8, 'subsample':0.8, 'colsample_bytree':0.7,
                         'alpha':0.1, 'lambda':1.0, 'tree_method':'hist'}),
    ]
    
    for cfg_name, cfg_params in xgb_configs:
        for s in ALL_SEEDS:
            p = cfg_params.copy(); p['seed'] = s
            train_and_cache(f"{cfg_name}_s{s}", get_xgb_trainer(p))
else:
    print("[PHASE 3] SKIPPED — XGBoost not installed")

## 🚀 8. Phase 4 — CatBoost (The Heart of Colossus)

**130 models** (13 configs × 10 seeds). Slowest phase, ~24-48 hours.  

This is the most critical phase. Our competition history proved definitively:
> **Deep CatBoost trees (depth 8-12) are the only models that satisfy the Explained Variance Penalty.**  
> When the stacker is CatBoost-dominant, LB scores improve significantly.

We sweep across:
- **Depths 3-12** — Full spectrum from conservative to highly expressive
- **Slow learning rate (0.01) variants** for depths 6, 8, 10 — More trees, more careful splits
- **High regularization (l2=10)** variants — Explore the regularization axis independently

CatBoost's **symmetric trees** (balanced splits at each level) differ fundamentally from LightGBM's leaf-wise growth, providing genuine algorithmic diversity at every depth.

In [ ]:
print("[PHASE 4] CatBoost — 130 models (HEAVY COMPUTE)")

cat_base = dict(iterations=5000, l2_leaf_reg=3.0, min_data_in_leaf=15,

                task_type='GPU', verbose=0,
                eval_metric='RMSE', early_stopping_rounds=150)

cat_configs = [
    # Depth sweep — standard LR
    ("cat_d3",       {**cat_base, 'depth':3,  'learning_rate':0.03}),
    ("cat_d4",       {**cat_base, 'depth':4,  'learning_rate':0.03}),
    ("cat_d5",       {**cat_base, 'depth':5,  'learning_rate':0.03}),
    ("cat_d6",       {**cat_base, 'depth':6,  'learning_rate':0.03}),
    ("cat_d7",       {**cat_base, 'depth':7,  'learning_rate':0.03}),
    ("cat_d8",       {**cat_base, 'depth':8,  'learning_rate':0.03}),
    ("cat_d10",      {**cat_base, 'depth':10, 'learning_rate':0.03}),
    ("cat_d12",      {**cat_base, 'depth':12, 'learning_rate':0.03}),
    # Slow LR variants — deeper learning with more trees
    ("cat_d6_slow",  {**cat_base, 'depth':6,  'learning_rate':0.01, 'iterations':10000}),
    ("cat_d8_slow",  {**cat_base, 'depth':8,  'learning_rate':0.01, 'iterations':10000}),
    ("cat_d10_slow", {**cat_base, 'depth':10, 'learning_rate':0.01, 'iterations':10000}),
    # High regularization variants — explore reg axis
    ("cat_d6_reg",   {**cat_base, 'depth':6,  'learning_rate':0.03, 'l2_leaf_reg':10.0}),
    ("cat_d8_reg",   {**cat_base, 'depth':8,  'learning_rate':0.03, 'l2_leaf_reg':10.0}),
]

for cfg_name, cfg_params in cat_configs:
    for s in ALL_SEEDS:
        p = cfg_params.copy(); p['random_seed'] = s
        train_and_cache(f"{cfg_name}_s{s}", get_cat_trainer(p))

## 🧮 9. Stacking & Final Submission

### Why HuberRegressor?

The **HuberRegressor** (ε=1.35) is the proven optimal meta-model for this competition's custom metric:

- Standard `Ridge` stacker: computes a weighted mean. If any single base model produces extreme predictions, they leak through.
- `HuberRegressor`: For any training sample where a base model's contribution differs from the truth by more than ε=1.35 standard deviations, the loss function *ignores that sample's gradient*. This mathematically prevents outlier-producing base models from getting large weights.
- The result: deep CatBoost models (high variance, good fluctuation tracking) get positive weights, while models that produce occasional spikes get their weights clamped.

> 💡 **Run this cell at any time** — it stacks whatever checkpoints are available in `colossus_checkpoints/`.

In [ ]:
import glob
from sklearn.linear_model import HuberRegressor

print("=" * 60)
print("  COLOSSUS AUTO-STACKER")
print("=" * 60)

oof_files   = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "*_oof.npy")))
model_names = [os.path.basename(f).replace("_oof.npy", "") for f in oof_files
               if "features_cached" not in f]

print(f"\nFound {len(model_names)} completed models")
families = {}
for m in model_names:
    families.setdefault(m.split('_')[0], []).append(m)
for fam, mems in sorted(families.items()):
    print(f"  {fam:5s} : {len(mems)} models")

oof_stack  = np.column_stack([np.load(os.path.join(CHECKPOINT_DIR, f"{m}_oof.npy"))  for m in model_names])
test_stack = np.column_stack([np.load(os.path.join(CHECKPOINT_DIR, f"{m}_test.npy")) for m in model_names])

print(f"\nStacking matrix: {oof_stack.shape[0]} samples × {oof_stack.shape[1]} models")

# HuberRegressor stacker
huber = HuberRegressor(epsilon=1.35, alpha=0.0001, max_iter=2000)
huber.fit(oof_stack, y_arr)

huber_oof  = huber.predict(oof_stack)
huber_test = huber.predict(test_stack)

rmse_final = np.sqrt(mean_squared_error(y_arr, huber_oof))
r2_final   = r2_score(y_arr, huber_oof)

print(f"\n[COLOSSUS STACK] OOF RMSE = {rmse_final:.5f} | R2 = {r2_final:.4f}")

# Top 20 model weights
weights = sorted(zip(model_names, huber.coef_), key=lambda x: abs(x[1]), reverse=True)
print(f"\nTop 20 model weights:")
for name, w in weights[:20]:
    print(f"  {name:28s} : {w:+.4f}")

# Save
os.makedirs("submissions", exist_ok=True)
preds_clipped = np.clip(huber_test, 0, 1)
sub = pd.DataFrame({'record_id': test_record_ids, 'flood_risk_score': preds_clipped})
sub.to_csv("submissions/submission_v20_colossus.csv", index=False)
print(f"\n✅ Saved: submissions/submission_v20_colossus.csv")

## 🎯 10. Blending with V18 Titan

Blending V20 with V18 reduces correlated errors — even if both models have weaknesses, they tend to fail in different regions.

We generate blends at **5 different ratios** so you can test multiple submissions across days:
- **30% V20 / 70% V18** — Conservative, leans heavily on proven champion
- **50% V20 / 50% V18** — Balanced blend
- **70% V20 / 30% V18** — Aggressive, when V20 is nearly complete

> 💡 **Strategy**: Submit the 50/50 blend early. Once CatBoost models dominate the stack, switch to 70% V20.

In [ ]:
v18_path = os.path.join("submissions", "submission_v18_titan.csv")

if os.path.exists(v18_path):
    v18 = pd.read_csv(v18_path)
    v18_preds = v18['flood_risk_score'].values
    
    print("Blends with V18 Titan:")
    for w20 in [0.3, 0.4, 0.5, 0.6, 0.7]:
        w18      = 1.0 - w20
        blended  = w18 * v18_preds + w20 * preds_clipped
        fname    = f"submissions/submission_v20_blend_{int(w20*100)}v20_{int(w18*100)}v18.csv"
        sub_b    = pd.DataFrame({'record_id': test_record_ids,
                                 'flood_risk_score': np.clip(blended, 0, 1)})
        sub_b.to_csv(fname, index=False)
        print(f"  ✅ {fname.split('/')[-1]}")
else:
    print(f"⚠️  V18 submission not found at {v18_path}")
    print("   Only standalone V20 submission generated.")

## 📋 11. Complete Model Count Summary

When all phases complete:

In [ ]:
hgb_count  = 4  * N_SEEDS  # 4 depth configs
lgb_count  = 6  * N_SEEDS  # 6 LGB configs
xgb_count  = 4  * N_SEEDS  # 4 XGB configs (if installed)
cat_count  = 13 * N_SEEDS  # 13 CatBoost configs
total      = hgb_count + lgb_count + (xgb_count if HAS_XGB else 0) + cat_count

print(f"""
╔══════════════════════════════════════════╗
  V20 COLOSSUS FINAL MODEL COUNT
╠══════════════════════════════════════════╣
  Phase 1 - HGB      : {hgb_count:>4} models  (4  cfg × {N_SEEDS} seeds)
  Phase 2 - LightGBM : {lgb_count:>4} models  (6  cfg × {N_SEEDS} seeds)
  Phase 3 - XGBoost  : {xgb_count if HAS_XGB else 0:>4} models  (4  cfg × {N_SEEDS} seeds)
  Phase 4 - CatBoost : {cat_count:>4} models  (13 cfg × {N_SEEDS} seeds)
────────────────────────────────────────────
  TOTAL               : {total:>4} model configs
  Folds per model     : {TOTAL_FOLDS}
  Total model fits    : {total * TOTAL_FOLDS:,}
╚══════════════════════════════════════════╝
""")

## 🥇 12. The Final Geometric Blend (The Ultimate Ensembling Strategy)

In the final hour of the competition, we fused the three historical champion architectures together:
1. **V17 Genesis** (`0.38163`) — The 20-fold meta-ensemble pioneer.
2. **V18 Titan** (`0.38149`) — The massive 36-model, 3-seed scale champion.
3. **V20 Colossus** (`0.38125`) — The 100-fold multi-GPU swarm.

Instead of a simple weighted average, we used a **Geometric Mean**. This mathematically penalizes extreme spikes more harshly than an arithmetic mean. Because the three models learned fundamentally distinct signals, the geometric blend cleanly squashed their respective outliers while perfectly preserving their underlying variance tracking, completely maximizing the Balanced Error Assessment metric.

In [ ]:
print("=== ML Opsidian Genesis: Final Geo-Blend ===")

v17_path = "submissions/submission_v17_genesis_ultimate.csv"
v18_path = "submissions/submission_v18_titan.csv"
v20_path = "submissions/submission_v20_colossus.csv"

if os.path.exists(v17_path) and os.path.exists(v18_path) and os.path.exists(v20_path):
    v17 = pd.read_csv(v17_path)
    v18 = pd.read_csv(v18_path)
    v20 = pd.read_csv(v20_path)

    # Geometric Mean (33% each)
    geo_blend = np.power(v17['flood_risk_score'] * v18['flood_risk_score'] * v20['flood_risk_score'], 1/3.0)

    sub = v17.copy()
    sub['flood_risk_score'] = geo_blend
    out_file = "submissions/submission_final_geo_blend.csv"
    sub.to_csv(out_file, index=False)

    print(f"✅ Saved final geo-blend submission: {out_file}")
    print("This achieved a top-tier score of 0.38130!")
else:
    print("⚠️ Missing historical submission files. Cannot compute Geo-Blend.")
